In [0]:
df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("/Volumes/workspace/ecommerce/ecommerce_data/2019-Oct.csv")

df_n = spark.read \
    .option("header",True) \
    .option("inferSchema",True) \
    .csv("/Volumes/workspace/ecommerce/ecommerce_data/2019-Nov.csv")

##DAY 0

In [0]:
!pip install kaggle

In [0]:
import os

os.environ["KAGGLE_USERNAME"] = "my3sheth"
os.environ["KAGGLE_KEY"] = "KGAT_0daf1c5f3e9ab021571cb8a8e91780ee"

print("Kaggle credentials configured!")

In [0]:
spark.sql("""
CREATE SCHEMA IF NOT EXISTS workspace.ecommerce
""")

In [0]:
spark.sql("""
CREATE VOLUME IF NOT EXISTS workspace.ecommerce.ecommerce_data
""")

In [0]:
%restart_python

###Run only if new data is to be loaded

In [0]:
# !pip install kagglehub

# import kagglehub

# # Download latest version
# path = kagglehub.dataset_download("mkechinov/ecommerce-behavior-data-from-multi-category-store")

# print("Path to dataset files:", path)

In [0]:
# import shutil

# path = "/home/spark-51af5c84-6847-4d06-870d-91/.cache/kagglehub/datasets/mkechinov/ecommerce-behavior-data-from-multi-category-store/versions/8"

# src_path = f"{path}/2019-Nov.csv"
# dst_path1 = "/Volumes/workspace/ecommerce/ecommerce_data/2019-Nov.csv"

# shutil.copy(src_path, dst_path1)

# dst_path2 = "/Volumes/workspace/ecommerce/ecommerce_data/2019-Oct.csv"

# shutil.copy(src_path, dst_path2)

In [0]:
print(f"October 2019 - Total Events: {df.count():,}")
print("\n" + "="*60)
print("SCHEMA:")
print("="*60)
df.printSchema()

In [0]:
print("\n" + "="*60)
print("SAMPLE DATA (First 5 rows):")
print("="*60)
df.show(5, truncate=False)

##DAY 1

In [0]:
# Create simple DataFrame
data = [("iPhone", 999), ("Samsung", 799), ("MacBook", 1299)]
df = spark.createDataFrame(data, ["product", "price"])
df.show()

# Filter expensive products
df.filter(df.price > 1000).show()

In [0]:
path = "/Volumes/workspace/ecommerce/ecommerce_data/2019-Oct.csv"

df = spark.read.option("header", True)\
               .option("inferSchema", True)\
               .csv(path)

print ("Oct 2019 metadata:\n")
df.printSchema()

print("\nOct 2019 data:\n")

df.show(5, truncate=True, vertical=False)

In [0]:
path_n = "/Volumes/workspace/ecommerce/ecommerce_data/2019-Nov.csv"

df_n = spark.read.option("header", True)\
               .option("inferSchema", True)\
               .csv(path_n)

print ("Nov 2019 metadata:\n")
df_n.printSchema()

print("\nNov 2019 data:\n")

df_n.show(5, truncate=True, vertical=False)

In [0]:
df.select("event_type", "price").show (5)

In [0]:
df_n.selectExpr("price * 2 as double_price").show(5)

##DAY 2

In [0]:
df.filter(df.price > 1000).show()

In [0]:
from pyspark.sql.functions import col

df.withColumn("tax_amount", col("price") * 0.18).show()

In [0]:
from pyspark.sql.functions import when, col

df.withColumn(
    "price_category",
    when(col("price") > 1000, "High").otherwise("Low")
).show(n=150)

In [0]:
display(df.orderBy(df.price.asc()).limit(5))

In [0]:
display(df.groupBy("event_type").count())

Databricks visualization. Run in Databricks to view.

In [0]:
top_brands = df.groupBy("brand").count().orderBy("count", ascending=False).limit(5)
display(top_brands)

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS ecommerce
COMMENT 'Schema for e-commerce analytics';

In [0]:
%sql
USE ecommerce;

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS ecommerce_data
COMMENT 'Volume to store raw e-commerce CSV data';

##DAY 3

In [0]:
print ("Original data:")
print(df.count())
print("\n")
print ("Records with unidentified brands:")
df2 = df.dropna(subset=["brand"])
print(df2.count())

In [0]:
df.select("user_id").distinct().count()

In [0]:
df.groupBy("user_id").count().count()

In [0]:
df.dropDuplicates(["product_id"]).count()

In [0]:
df.join(df_n, on="user_id", how="inner").show(vertical=True)

In [0]:
df.join(
    df_n,
    (df.user_id == df_n.user_id) &
    (df.event_time == df_n.event_time),
    how="inner"
).show()

In [0]:
from pyspark.sql.functions import broadcast

df_subset = df.limit (5)

df.join(
    broadcast(df_subset),
    "user_id",
    "inner"
).show(vertical=True)


In [0]:
from pyspark.sql import Window
from pyspark.sql.functions import lag

window_spec = Window.partitionBy("user_id").orderBy("event_time") 

df_with_prev_price = df.withColumn(
    "prev_price",
    lag("price", 1).over(window_spec)
)

display(df_with_prev_price)

In [0]:
def price_category(price):
    if price is None:
        return "UNKNOWN"
    elif price > 1000:
        return "HIGH"
    else:
        return "LOW"

In [0]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

price_category_udf = udf(price_category, StringType())

In [0]:
df.withColumn(
    "price_category",
    price_category_udf(df.price)
).show()

##DAY 4

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.ecommerce.delta;

In [0]:
df.write.format("delta") \
  .mode("overwrite") \
  .option("mergeSchema", "true")\
  .save("/Volumes/workspace/ecommerce/delta/events")

In [0]:
df.write \
  .format("delta") \
  .mode("append") \
  .option("mergeSchema", "true")\
  .saveAsTable("ecommerce.delta")

In [0]:
spark.sql("""
    CREATE OR REPLACE TABLE ecommerce.delta_events
    USING DELTA
    AS SELECT * FROM ecommerce.delta
""")

# Test schema enforcement
try:
    wrong_schema = spark.createDataFrame([("a","b","c")], ["x","y","z"])
    wrong_schema.write.format("delta").mode("append").save("/delta/events")
except Exception as e:
    print(f"Schema enforcement: {e}")

In [0]:
%sql
SELECT * FROM workspace.ecommerce.delta_events limit 15;

##DAY 5

In [0]:
%sql
DESCRIBE HISTORY workspace.ecommerce.delta_events;

In [0]:
%sql
SELECT * 
FROM workspace.ecommerce.events VERSION AS OF 0;

In [0]:
df.createOrReplaceTempView("staging_events")

In [0]:
%sql
OPTIMIZE workspace.ecommerce.events
ZORDER BY (user_id, event_time);

In [0]:
%sql
VACUUM workspace.ecommerce.events RETAIN 168 HOURS;

##DAY 6

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.bronze
COMMENT 'Bronze layer: raw ingested data (append-only)';

CREATE SCHEMA IF NOT EXISTS workspace.silver
COMMENT 'Silver layer: cleaned, validated, deduplicated data';

CREATE SCHEMA IF NOT EXISTS workspace.gold
COMMENT 'Gold layer: business-ready aggregates and KPIs';

In [0]:
from pyspark.sql.functions import current_date

df_raw = spark.read \
    .option("header", True) \
    .csv("/Volumes/workspace/ecommerce/ecommerce_data/*.csv")

df_bronze = df_raw \
    .withColumn("_ingest_date", current_date()) \
    .withColumn("_source_file", df_raw["_metadata.file_path"])

In [0]:
df_bronze.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("workspace.bronze.ecommerce_events")

In [0]:
%sql
SELECT * FROM workspace.bronze.ecommerce_events LIMIT 10;

In [0]:
from pyspark.sql.functions import col, lower, row_number, floor
from pyspark.sql.window import Window

df_bronze = spark.table("workspace.bronze.ecommerce_events")

df_bronze = df_bronze.withColumn(
    "price_double",
    col("price").cast("double")
)

In [0]:
window = Window.partitionBy("user_id", "event_time") \
               .orderBy(col("_ingest_date").desc())

df_silver = df_bronze \
    .filter(col("price_double").isNotNull() & (col("price_double") > 0)) \
    .withColumn("brand", lower(col("brand"))) \
    .withColumn("rn", row_number().over(window)) \
    .filter(col("rn") == 1) \
    .drop("rn")

In [0]:
from pyspark.sql.functions import floor

df_silver = df_silver.withColumn(
    "price_int",
    floor(col("price_double")).cast("bigint")
)

In [0]:
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.silver.ecommerce_events")

In [0]:
%sql
SELECT COUNT(*) as BRONZE_LAYER FROM workspace.bronze.ecommerce_events;

In [0]:
%sql
SELECT COUNT(*) as SILVER_LAYER FROM workspace.silver.ecommerce_events;

In [0]:
from pyspark.sql.functions import sum, to_date

df_gold = spark.table("workspace.silver.ecommerce_events") \
    .filter(col("event_type") == "purchase") \
    .withColumn("event_date", to_date("event_time")) \
    .groupBy("event_date") \
    .agg(sum("price").alias("daily_revenue"))

In [0]:
df_gold.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.daily_revenue")

In [0]:
%sql
SELECT COUNT(*) as GOLD_LAYER FROM workspace.gold.daily_revenue;

##DAY 7

In [0]:
dbutils.widgets.text("process_date", "")
process_date = dbutils.widgets.get("process_date")

In [0]:
df_p = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv(f"/Volumes/workspace/ecommerce/raw/events/process_date={process_date}/*")


In [0]:
file_map = {
  "2019-10": "2019-Oct.csv",
  "2019-11": "2019-Nov.csv",
  # Add as you go
}

file_name = file_map.get(process_date)
if not file_name:
    raise Exception(f"No file for process_date={process_date}")

path = f"/Volumes/workspace/ecommerce/ecommerce_data/{file_name}"

df_p2 = spark.read.option("header", True).csv(path)


In [0]:
df_p2.show(5)

##DAY 8

In [0]:
%sql
CREATE CATALOG ecommerce;

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS ecommerce_catalog;

CREATE SCHEMA IF NOT EXISTS ecommerce_catalog.sales;

CREATE SCHEMA IF NOT EXISTS ecommerce_catalog.marketing;

In [0]:
df.write.format("delta").mode("overwrite").saveAsTable("ecommerce_catalog.sales.events_oct")

In [0]:
%sql
GRANT SELECT ON SCHEMA ecommerce_catalog.sales TO `maitrisheth09@gmail.com`;

In [0]:
%sql
REVOKE SELECT ON TABLE ecommerce_catalog.sales.events_oct FROM `maitrisheth09@gmail.com`;

In [0]:
%sql
select * from ecommerce_catalog.sales.events_oct


In [0]:
%sql
CREATE VIEW ecommerce_catalog.sales.product_view AS
SELECT category_id, brand, price
FROM ecommerce_catalog.sales.events_oct
WHERE brand != TRUE;

GRANT SELECT ON VIEW ecommerce_catalog.sales.product_view TO `maitrisheth09@gmail.com`;